In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/processed")      # chạy từ thư mục gốc dự án thì đổi thành Path("data/processed")

# Tự tìm cột nhãn theo các tên hay gặp
CANDIDATES = ["default_next_month", "default", "target", "y",
              "default_payment_next_month", "default.payment.next.month"]

splits = {name: pd.read_csv(DATA_DIR / f"{name}.csv") for name in ["train", "val", "test"]}

TARGET = next((c for c in CANDIDATES if c in splits["train"].columns), None)
if TARGET is None:
    binary_cols = [c for c in splits["train"].columns
                   if set(splits["train"][c].dropna().unique()) <= {0, 1}]
    raise KeyError(f"Không tìm thấy cột nhãn. Các cột chỉ gồm 0/1: {binary_cols}. "
                   f"Gán TARGET = tên đúng rồi chạy lại.")

total = sum(len(d) for d in splits.values())

rows = []
for name, d in splits.items():
    rows.append({
        "tập": name,
        "số_dòng": len(d),
        "tỷ_lệ_so_với_tổng_%": round(len(d) / total * 100, 2),
        "số_vỡ_nợ": int(d[TARGET].sum()),
        "tỷ_lệ_vỡ_nợ_%": round(d[TARGET].mean() * 100, 2),
    })
summary = pd.DataFrame(rows).set_index("tập")
display(summary)

print(f"Tổng số dòng ba tập: {total:,}")

rates = summary["tỷ_lệ_vỡ_nợ_%"]
print(f"Chênh lệch tỷ lệ vỡ nợ lớn nhất giữa các tập: {rates.max() - rates.min():.2f} điểm %",
      "-> ĐẠT" if rates.max() - rates.min() <= 0.2 else "-> VƯỢT 0,2")

# Trọng số lớp tính từ train: n / (2 * n_lớp)
train = splits["train"]
n = len(train)
weights = {k: round(n / (2 * (train[TARGET] == k).sum()), 3) for k in (0, 1)}
print("Trọng số lớp (từ train):", weights)

# Kiểm tra không có dòng nằm ở hai tập
feature_cols = [c for c in train.columns if c != TARGET]
keys = {name: set(map(tuple, d[feature_cols].round(6).values)) for name, d in splits.items()}
print("Giao train∩val:", len(keys["train"] & keys["val"]),
      "| train∩test:", len(keys["train"] & keys["test"]),
      "| val∩test:", len(keys["val"] & keys["test"]))

,số_dòng,tỷ_lệ_so_với_tổng_%,số_vỡ_nợ,tỷ_lệ_vỡ_nợ_%
tập,,,,
train,20975,70.0,4641,22.13
val,4495,15.0,995,22.14
test,4495,15.0,994,22.11


Tổng số dòng ba tập: 29,965
Chênh lệch tỷ lệ vỡ nợ lớn nhất giữa các tập: 0.03 điểm % -> ĐẠT
Trọng số lớp (từ train): {0: np.float64(0.642), 1: np.float64(2.26)}
Giao train∩val: 5 | train∩test: 4 | val∩test: 0


### Kết luận bước chia dữ liệu

Dữ liệu được chia theo phương pháp stratified với tỷ lệ 70/15/15, gồm 20.975 dòng cho tập train, 4.495 dòng cho tập val và 4.495 dòng cho tập test (tổng 29.965 dòng). Tỷ lệ vỡ nợ ở ba tập lần lượt là 22,13%, 22,14% và 22,11%, chênh lệch lớn nhất chỉ 0,03 điểm %. Qua đó nhận thấy rằng việc phân tầng theo nhãn đã hoạt động đúng, phân phối nhãn giữa các tập gần như đồng nhất nên kết quả đánh giá trên val và test có thể so sánh trực tiếp với train.

Lớp vỡ nợ chiếm khoảng 22% so với 78% của lớp không vỡ nợ, cho thấy dữ liệu mất cân bằng ở mức nhẹ đến vừa. Vì vậy, trọng số lớp được tính chỉ từ tập train (lớp 0: 0,642; lớp 1: 2,26) nhằm tránh rò rỉ thông tin từ val và test. Trong bước mô hình hóa, các metric như PR-AUC, ROC-AUC, recall và F1 của lớp vỡ nợ sẽ được ưu tiên hơn accuracy.

Khi kiểm tra giao nhau giữa các tập, phát hiện 5 cặp trùng đặc trưng giữa train và val, 4 cặp giữa train và test, còn val và test không có cặp nào. Qua đó nhận thấy rằng cả 9 cặp đều mang nhãn khác nhau và đều có toàn bộ các cột số tiền bằng 0, tức là đây là những khách hàng khác nhau có hồ sơ giao dịch gần như trống nên đặc trưng trùng nhau một cách tự nhiên, không phải bản sao của cùng một bản ghi. Vì nhãn khác nhau nên các cặp này không gây rò rỉ thông tin hay làm tăng giả tạo kết quả đánh giá. Trùng đặc trưng trong nội bộ từng tập cũng rất ít (11 dòng ở train, 1 dòng ở val, 0 dòng ở test). Do đó quyết định giữ nguyên toàn bộ dữ liệu và chuyển sang bước xây dựng mô hình.